# Formatting conversations for FOMO AI

This original notebook shows how the FOMO AI Brain reference implementation turns a request into structured messages, keeps a short conversation, and optionally asks a **real local model** for an answer.

There is no public FOMO AI API in this example. The code does not claim to be a newly trained model or reproduce ChatGPT internals.

## 1. Load the FOMO AI Brain package

Start Jupyter from the cloned repository. No Python dependencies are needed for the formatting cells.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'fomo_brain').is_dir():
    root = root.parent
if not (root / 'fomo_brain').is_dir():
    raise RuntimeError('Open this notebook from the fomo-ai-brain repository')
sys.path.insert(0, str(root))

from fomo_brain import FomoBrain, Message, ModelUnavailable, OllamaProvider, format_messages

## 2. Format a single request

A `system` message describes the assistant's behavior. The user's actual request follows it. Every message has an explicit role and non-empty text.

In [ ]:
request = format_messages('Explain how a cache helps a website.')
for message in request:
    print(message.as_dict())

## 3. Add previous turns

Pass only completed `user` and `assistant` messages as history. The next user message is appended after them. Do not put untrusted text in a new `system` message.

In [ ]:
history = [
    Message('user', 'What is a cache?'),
    Message('assistant', 'A cache stores recently used data for faster access.'),
]
follow_up = format_messages('When should I clear it?', history)
print([message.role for message in follow_up])
print(follow_up[-1].content)

## 4. Reject malformed input

Inputs are validated before a model call. Failed model calls also leave the stored conversation unchanged.

In [ ]:
try:
    format_messages('   ')
except ValueError as error:
    print('Rejected:', error)

## 5. Optional: generate a real answer locally

This cell calls Ollama on your own computer. First run `ollama pull qwen3:0.6b` and `ollama serve` outside Jupyter. If the model is unavailable, the cell raises an explicit error; it does **not** display a fabricated answer. No cloud API key is needed.

In [ ]:
brain = FomoBrain(OllamaProvider(model='qwen3:0.6b'), max_turns=6)
print(brain.ask('Explain what a cache is in two short sentences.'))
print(brain.ask('Give me one concrete example.'))
print('Saved messages:', len(brain.history))